# RSA

RSA (Rivest–Shamir–Adleman) is an algorithm used by modern computers to encrypt and decrypt messages. It is an asymmetric cryptographic algorithm. Asymmetric means that there are two different keys. This is also called public key cryptography, because one of the keys can be given to anyone. The other key must be kept private. The algorithm is based on the fact that finding the factors of a large composite number is difficult: when the factors are prime numbers, the problem is called prime factorization. It is also a key pair (public and private key) generator.

RSA involves a public key and private key. The public key can be known to everyone- it is used to encrypt messages. Messages encrypted using the public key can only be decrypted with the private key. The private key needs to be kept secret. Calculating the private key from the public key is very difficult.

## Modular Arithmetic

### 4:3 = 1 remainder 1;  7:3 = 2 remainder 1 ->  4 = 7 mod 3

## Euler theorem

## n = p q
## $\Phi$(n) = (p-1)* (q-1)
## $\forall$ a<n : (a,  n) = 1
## $a^{\bf  \Phi(n)}$ = 1 mod n
## e < n, (e, n) = 1,  (e, $\Phi$(n)) = 1
## $\exists$ d < n : ed = 1 mod $\Phi$(n)
## that is:
## e*d + $\Phi$(n) * k = 1

## Generating keys

<div>
<img src="./img/Keys.png" width="800" align="left"  /img>
</div>

## Encrypting message

<div>
<img src="./img/Crypting.png" width="1200" align="left"  /img>
</div>

## Decrypting message

<div>
<img src="./img/Decrypting.png" width="1200" align="left"  /img>
</div>

## Example

<div>
<img src="./img/Example-1.png" width="1200" align="left"  /img>
</div>

## Crypting and Decrypting a message m

<div>
<img src="./img/Crypting-Decrypting-example.png" width="800" align="left"  /img>
</div>

In [1]:
import math        # math.gcd = greatest common divisor
import random      # (not used below; handy for picking random primes)


In [2]:
# modulus (remainder) operator -> %
# a^m mod n -> pow(a, m, n)   (3-argument pow: fast modular exponentiation, numbers never grow large)
#examples
print(5%4)          # 1
print(9%4)          # 1
print(pow(2, 5,7))  # 2^5 = 32, 32 mod 7 = 4


1
1
4


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `a % n` | the remainder of the integer division: the 'mod' of modular arithmetic |
| `pow(a, m, n)` | aᵐ mod n by repeated squaring: the numbers never exceed n², so it works for 2048-bit exponents |

</details>


In [3]:
p=61            # first prime (secret)
q=53            # second prime (secret)
N = p*q         # the public modulus
print(p,q, N)


61 53 3233


In [4]:
def phi(p,q):
    """Euler's totient of N = p*q for primes p, q."""
    assert(math.gcd(p,q) == 1), "p e q non sono primi fra loro"   # assert: stop with this message if False
    return (p-1)*(q-1)

def modinverse(e, phi):
    """d such that e*d = 1 (mod phi), found by brute force (fine for small numbers)."""
    e = e % phi
    for x in range(1, phi):           # try every candidate x = 1, 2, ..., phi-1
        if ((e * x) % phi == 1):
            return x
    return 1

def encrypt(m, e, n):
    return pow(m, e, n)               # c = m^e mod n

def decrypt(c, d, n):
    return pow(c, d, n)               # m = c^d mod n


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `p, q` | two primes, kept secret; `N = p*q` is public and factoring it is the whole difficulty of RSA |
| `phi(p, q) = (p-1)*(q-1)` | Euler's totient of N = pq: the number of integers below N that are coprime with N |
| `modinverse(e, phi)` | brute force over all x < φ(N): fine for 3120, hopeless for a real key (the exercises replace it with Euclid) |
| `encrypt` / `decrypt` | the same `pow` with the public exponent e or the private exponent d |
| `assert cond, msg` | a guard: stops with `msg` if the condition is False |

</details>


In [5]:
modulus = phi (p,q)              # phi(N) = (p-1)(q-1) = 3120
print("il modulo è:",modulus)


il modulo è: 3120


In [6]:
e = 17                           # public exponent: must be coprime with phi(N) (real systems use 65537)
print("la chiave pubblica è:", e)


la chiave pubblica è: 17


In [7]:
d = modinverse(e, modulus)       # private exponent d = e^{-1} mod phi(N)
print("la chiave privata è:", d)


la chiave privata è: 2753


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `e = 17` | the public exponent: any number coprime with φ(N); real systems use 65537 = 2¹⁶ + 1 (two 1-bits, hence 16 squarings and one multiplication) |
| `d = modinverse(e, modulus)` | the private exponent, defined by e·d ≡ 1 (mod φ(N)); only whoever knows φ(N), i.e. p and q, can compute it |
| `e*d % 3120 == 1` | the check of the definition: 17 · 2753 = 46801 = 15 · 3120 + 1 |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**The private exponent by the extended Euclidean algorithm** (what `modinverse` finds by trying all 3120 candidates). Divide $3120$ by $17$ and continue with the remainders:

$$3120 = 183\cdot 17 + 9,\qquad 17 = 1\cdot 9 + 8,\qquad 9 = 1\cdot 8 + 1 .$$

The last non-zero remainder is $1$, so $\gcd(17, 3120) = 1$ and the inverse exists. Now walk back up, writing $1$ as a combination of $3120$ and $17$:

$$1 = 9 - 8 = 9 - (17 - 9) = 2\cdot 9 - 17 = 2\,(3120 - 183\cdot 17) - 17 = 2\cdot 3120 - 367\cdot 17 .$$

Hence $17\cdot(-367) \equiv 1 \pmod{3120}$, and $d = -367 + 3120 = 2753$ ✓ — three divisions instead of 2753 trials. For a 2048-bit key the brute force would need about $10^{616}$ trials; Euclid needs a few thousand divisions.

**The message.** $c = 123^{17} \bmod 3233 = 855$ and $855^{2753} \bmod 3233 = 123$: both are computed by `pow` with repeated squaring (17 = 10001₂: four squarings and one multiplication for the encryption).

</details>


In [8]:
print("il prodotto di e per d è:", e*d)
print("il modulo di e*d rispetto a 3120 è:", e*d % 3120)   # must be 1: that is what makes decryption work


il prodotto di e per d è: 46801
il modulo di e*d rispetto a 3120 è: 1


In [9]:
message = 123                    # the plaintext must be an integer < N
print("il messaggio da criptare è:",message )


il messaggio da criptare è: 123


In [10]:
encrypted = encrypt(message, e, N)      # anyone can do this with the public key (N, e)
print ("The ciphertext generated is ", encrypted)


The ciphertext generated is  855


In [11]:
plaintext = decrypt(encrypted, d, N)    # only the owner of d can do this
print("The plain text is; ", plaintext)


The plain text is;  123


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `message = 123` | the plaintext must be an integer smaller than N; a text is first turned into such integers (exercise 3) |
| `encrypt(message, e, N)` | c = mᵉ mod N: anyone can do this, the key (N, e) is public |
| `decrypt(encrypted, d, N)` | m = cᵈ mod N: Euler's theorem makes m^{ed} ≡ m because ed ≡ 1 (mod φ(N)) |

</details>


## 🔬 In-depth analysis

1. **A faster inverse.** `modinverse` tries every $x$ below $\varphi(N)$: 3120 steps here, about $10^{616}$ for a real 2048-bit key. Write `egcd(a, b)` (extended Euclid, returning $g, x, y$ with $ax + by = g$), derive $d$ from it and compare with Python's built-in `pow(e, -1, phi)`.
2. **A real-size exponent.** Take $p = 1\,000\,003$ and $q = 1\,000\,033$ (both prime) with $e = 65537$; encrypt and decrypt $m = 123\,456$. Why is `modinverse` hopeless here, while `egcd` is instantaneous?
3. **Text.** Encrypt the string `"QC"` by turning it into an integer with `int.from_bytes(b"QC", "big")` and back with `.to_bytes(2, "big")` (with the key of item 2, whose $N$ is larger than any 2-byte integer).
4. **Low exponent.** With $p = 101$, $q = 113$ and $e = 3$ (check $\gcd(3, \varphi(N)) = 1$), encrypt $m = 20$. Since $m^3 < N$, the ciphertext is $m^3$ *without* any modular reduction: recover $m$ from $c$ alone with an integer cube root — the attack that makes textbook RSA with $e = 3$ unsafe for small messages.

Write your code in the empty cell below; a reference solution follows it. Try first, then compare.


In [12]:
# 🔬 In-depth analysis — your code here



In [13]:
# Reference solution
# (1) extended Euclid: g = gcd(a, b) and the Bézout coefficients x, y with a*x + b*y = g
def egcd(a, b):
    if b == 0: return a, 1, 0
    g, x, y = egcd(b, a % b)                     # recursion on (b, a mod b): the same steps as the box above, bottom-up
    return g, y, x - (a // b) * y
g, x, y = egcd(e, modulus); d_euclid = x % modulus       # x may be negative (-367): % brings it into 0 … φ-1
print(f"(1) egcd: {e}*({x}) + {modulus}*({y}) = {g}  ->  d = {d_euclid}  (brute force gave {d}, pow(e, -1, phi) gives {pow(e, -1, modulus)})")

# (2) a key with million-sized primes and the standard exponent
p2, q2, e2 = 1_000_003, 1_000_033, 65537               # underscores are digit separators, allowed in Python literals
N2 = p2 * q2; phi2 = (p2 - 1) * (q2 - 1); d2 = pow(e2, -1, phi2)
m2 = 123_456; c2 = pow(m2, e2, N2)
print(f"(2) N = {N2}, phi = {phi2}, d = {d2}; c = {c2}, decrypted = {pow(c2, d2, N2)}  (modinverse would loop up to {phi2:.1e} times)")

# (3) text -> integer -> cipher -> integer -> text
m3 = int.from_bytes(b"QC", "big")                       # "big" = most significant byte first: 'Q'*256 + 'C'
c3 = pow(m3, e2, N2); back = pow(c3, d2, N2).to_bytes(2, "big").decode()
print(f"(3) 'QC' -> {m3} -> cipher {c3} -> {back!r}")

# (4) e = 3 on a small message: the cube root attack
p4, q4, e4 = 101, 113, 3; N4 = p4 * q4; phi4 = (p4 - 1) * (q4 - 1)
assert math.gcd(e4, phi4) == 1
m4 = 20; c4 = pow(m4, e4, N4)                            # m^3 = 8000 < N = 11413: no reduction happened
root = round(c4 ** (1 / 3)); root = next(r for r in (root - 1, root, root + 1) if r ** 3 == c4)   # exact integer cube root, checked on the neighbours of the float estimate
print(f"(4) e = 3: c = {c4} = m^3 exactly, cube root = {root} = m  (no private key needed)")


(1) egcd: 17*(-367) + 3120*(2) = 1  ->  d = 2753  (brute force gave 2753, pow(e, -1, phi) gives 2753)
(2) N = 1000036000099, phi = 1000034000064, d = 983264276609; c = 915632893244, decrypted = 123456  (modinverse would loop up to 1.0e+12 times)
(3) 'QC' -> 20803 -> cipher 896582680593 -> 'QC'
(4) e = 3: c = 8000 = m^3 exactly, cube root = 20 = m  (no private key needed)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `egcd(b, a % b)` | Euclid's recursion: gcd(a, b) = gcd(b, a mod b); unwinding it rebuilds the Bézout coefficients |
| `pow(e, -1, phi)` | Python ≥ 3.8 computes modular inverses natively: the reference for our `egcd` |
| `1_000_003` | underscores as digit separators in numeric literals |
| `int.from_bytes(b, "big")`, `.to_bytes(2, "big")` | bytes ↔ integer: the standard bridge between text and the integers RSA works on |
| `round(c ** (1/3))` + neighbours | a float cube root is only approximate: the integer check on r−1, r, r+1 makes it exact |

</details>
